In [0]:
# ============================================
# GOLD LAYER NOTEBOOK - CLEAN VERSION
# Keeps primary keys as-is
# KPI tables are lean
# Writes physical Delta files to ADLS
# Registers Databricks external tables on top
# ============================================

from pyspark.sql.functions import (
    col, lit, to_date, current_timestamp, countDistinct,
    sum as Fsum, avg as Favg, max as Fmax, when,
    year, month, dayofmonth
)

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ============================================
# CONFIG
# ============================================
catalog = "eus-sales-catalog"
silver_schema = "silver-layer"
gold_schema = "gold-layer"

# ============================================
# STORAGE PATHS
# ============================================
silver_orders_path = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/orders"

gold_orders_path   = "abfss://gold-layer@steusadlesgen0526.dfs.core.windows.net/orders"
daily_sales_path   = "abfss://gold-layer@steusadlesgen0526.dfs.core.windows.net/daily_sales_kpi"
product_kpi_path   = "abfss://gold-layer@steusadlesgen0526.dfs.core.windows.net/product_kpi"
customer_kpi_path  = "abfss://gold-layer@steusadlesgen0526.dfs.core.windows.net/customer_kpi"
region_kpi_path    = "abfss://gold-layer@steusadlesgen0526.dfs.core.windows.net/region_kpi"

# ============================================
# HELPERS
# ============================================
def save_and_register(df, path, table_name):
    """
    Write the Delta files first, then register the external table in UC.
    This avoids empty/stale table metadata issues.
    """
    full_table_name = f"`{catalog}`.`{gold_schema}`.`{table_name}`"

    # write physical delta files
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)

    # refresh catalog table registration
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
    spark.sql(f"""
    CREATE TABLE {full_table_name}
    USING DELTA
    LOCATION '{path}'
    """)
    spark.sql(f"REFRESH TABLE {full_table_name}")

# ============================================
# CREATE GOLD SCHEMA
# ============================================
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{gold_schema}`
""")

# ============================================
# READ SILVER FACT TABLE
# ============================================
fact_orders = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
    .withColumn("order_date", to_date(col("order_date")))
)

# ============================================
# READ DIMENSIONS (LATEST SCD2 CURRENT ROWS)
# ============================================
dim_customers_raw = spark.table(f"`{catalog}`.`{silver_schema}`.`customers`")
dim_products_raw  = spark.table(f"`{catalog}`.`{silver_schema}`.`products`")
dim_regions_raw   = spark.table(f"`{catalog}`.`{silver_schema}`.`regions`")
dim_stores_raw    = spark.table(f"`{catalog}`.`{silver_schema}`.`stores`")

current_customers = (
    dim_customers_raw
    .filter(col("IsCurrent") == 1)
    .select(
        col("customer_id"),
        col("skey_customer_id"),
        col("full_name")
    )
)

current_products = (
    dim_products_raw
    .filter(col("IsCurrent") == 1)
    .select(
        col("product_id"),
        col("skey_product_id"),
        col("product_name"),
        col("category")
    )
)

dim_regions = (
    dim_regions_raw
    .select(
        col("region_id"),
        col("region").alias("region_name")
    )
)

dim_stores = (
    dim_stores_raw
    .select(
        col("store_id"),
        col("store_name")
    )
)

# ============================================
# JOIN FACT + DIMENSIONS
# ============================================
joined_df = (
    fact_orders.alias("f")
    .join(
        current_customers.alias("c"),
        col("f.customer_id") == col("c.customer_id"),
        "left"
    )
    .join(
        current_products.alias("p"),
        col("f.product_id") == col("p.product_id"),
        "left"
    )
    .join(
        dim_regions.alias("r"),
        col("f.region_id") == col("r.region_id"),
        "left"
    )
    .join(
        dim_stores.alias("s"),
        col("f.store_id") == col("s.store_id"),
        "left"
    )
)

# ============================================
# GOLD DETAIL TABLE
# ============================================
gold_orders = (
    joined_df
    .select(
        # FACT
        col("f.order_id").alias("order_id"),
        col("f.order_date").alias("order_date"),
        col("f.customer_id").alias("customer_id"),
        col("f.product_id").alias("product_id"),
        col("f.store_id").alias("store_id"),
        col("f.region_id").alias("region_id"),
        col("f.quantity").alias("quantity"),
        col("f.price").alias("price"),
        col("f.item_amount").alias("item_amount"),
        col("f.total_amount").alias("total_amount"),
        col("f.is_dirty").alias("is_dirty"),
        col("f.ingestion_time").alias("ingestion_time"),

        # CUSTOMER
        col("c.skey_customer_id").alias("skey_customer_id"),
        col("c.full_name").alias("customer_name"),

        # PRODUCT
        col("p.skey_product_id").alias("skey_product_id"),
        col("p.product_name").alias("product_name"),
        col("p.category").alias("category"),

        # REGION / STORE
        col("r.region_name").alias("region_name"),
        col("s.store_name").alias("store_name")
    )
    .withColumn("gross_sales", col("quantity") * col("price"))
    .withColumn(
        "discount_amount",
        when(col("gross_sales") > 10000, col("gross_sales") * 0.10).otherwise(lit(0))
    )
    .withColumn("net_sales", col("gross_sales") - col("discount_amount"))
    .withColumn(
        "sales_category",
        when(col("net_sales") < 1000, "LOW")
        .when(col("net_sales") < 10000, "MEDIUM")
        .otherwise("HIGH")
    )
    .withColumn("order_year", year(col("order_date")))
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_day", dayofmonth(col("order_date")))
    .withColumn("gold_load_time", current_timestamp())
)

# ============================================
# KPI TABLES (LEAN)
# Only keep key + measures
# ============================================
daily_sales = (
    gold_orders
    .groupBy("order_date", "region_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        Fsum("quantity").alias("total_quantity"),
        Fsum("gross_sales").alias("gross_sales"),
        Fsum("discount_amount").alias("discount_amount"),
        Fsum("net_sales").alias("net_sales"),
        Favg("net_sales").alias("avg_sales")
    )
)

product_kpi = (
    gold_orders
    .groupBy("product_id")
    .agg(
        Fsum("quantity").alias("units_sold"),
        Fsum("gross_sales").alias("gross_sales"),
        Fsum("discount_amount").alias("discount_amount"),
        Fsum("net_sales").alias("total_sales"),
        Favg("net_sales").alias("avg_sales")
    )
)

customer_kpi = (
    gold_orders
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("orders"),
        Fsum("gross_sales").alias("gross_sales"),
        Fsum("discount_amount").alias("discount_amount"),
        Fsum("net_sales").alias("customer_lifetime_value"),
        Favg("net_sales").alias("avg_order_value"),
        Fmax("order_date").alias("last_order_date")
    )
)

region_kpi = (
    gold_orders
    .groupBy("region_id")
    .agg(
        countDistinct("order_id").alias("orders"),
        countDistinct("customer_id").alias("customers"),
        Fsum("quantity").alias("units_sold"),
        Fsum("gross_sales").alias("gross_sales"),
        Fsum("discount_amount").alias("discount_amount"),
        Fsum("net_sales").alias("region_sales"),
        Favg("net_sales").alias("avg_sales")
    )
)

# ============================================
# QUICK DEBUG COUNTS
# ============================================
# print("fact_orders count   =", fact_orders.count())
# print("current_customers   =", current_customers.count())
# print("current_products    =", current_products.count())
# print("dim_regions         =", dim_regions.count())
# print("dim_stores          =", dim_stores.count())
# print("gold_orders         =", gold_orders.count())
# print("daily_sales         =", daily_sales.count())
# print("product_kpi         =", product_kpi.count())
# print("customer_kpi        =", customer_kpi.count())
# print("region_kpi          =", region_kpi.count())

# Uncomment if you want to inspect sample rows
# display(gold_orders.limit(10))
# display(product_kpi.limit(10))
# display(customer_kpi.limit(10))
# display(region_kpi.limit(10))

# ============================================
# WRITE + REGISTER GOLD TABLES
# ============================================
save_and_register(gold_orders, gold_orders_path, "gold_orders")
save_and_register(daily_sales, daily_sales_path, "daily_sales_kpi")
save_and_register(product_kpi, product_kpi_path, "product_kpi")
save_and_register(customer_kpi, customer_kpi_path, "customer_kpi")
save_and_register(region_kpi, region_kpi_path, "region_kpi")

print("✅ GOLD LAYER COMPLETED")
print(f"Gold orders saved to: {gold_orders_path}")
print(f"Daily sales saved to: {daily_sales_path}")
print(f"Product KPI saved to: {product_kpi_path}")
print(f"Customer KPI saved to: {customer_kpi_path}")
print(f"Region KPI saved to: {region_kpi_path}")

In [0]:
# df = spark.read.format("delta").option("header", "true").load("abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/orders/").display()